# 🎮 AGAR-RL : Hub d'Évaluation & Replay HD (Google Drive)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Albin0903/agario/blob/main/notebooks/eval_drive_models.ipynb)

Ce notebook est dédié à **l'inspection, l'évaluation scientifique et la visualisation** de vos modèles d'IA entraînés stockés sur Google Drive :
1. **Connexion Drive & Inventaire** : scanne et répertorie tous vos modèles (`.zip` et `.onnx`) avec leurs paliers d'étapes (15M, final, etc.), tailles et dates.
2. **Évaluation Scientifique (100% même environnement)** : évalue le modèle dans l'environnement physique rigoureusement identique à l'entraînement (`AgarEnv`, 5 bots, lissage $\alpha=0.70$, masquage de split sous 36 de masse).
3. **Replay HD 80s** : génère et affiche le match vidéo HD avec affichage tête haute (HUD), vecteurs d'action et radar minimap.
4. **Export ONNX** : exporte le modèle sélectionné vers le standard universel ONNX.

## 0. Montage de Google Drive
Connexion sécurisée pour accéder à vos dossiers de sauvegarde `agario_rl_backup_v2`.

In [ ]:
import os, sys, time
from google.colab import drive

# 1. Montage Google Drive
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/agario_rl_backup_v2'
print(f'✅ Google Drive connecté.')
print(f'📁 Dossier cible : {DRIVE_DIR} (Existe: {os.path.exists(DRIVE_DIR)})')


## 1. Synchronisation Propre du Code GitHub & Installation

In [ ]:
import os

# Synchronisation forcée (reset hard) pour garantir l'alignement strict avec origin/main
if os.path.exists('.git'):
    print('🔄 Réinitialisation propre et synchronisation avec GitHub main...')
    !git fetch origin main
    !git reset --hard origin/main
elif os.path.exists('agario/.git'):
    print('🔄 Déplacement dans agario et synchronisation...')
    %cd agario
    !git fetch origin main
    !git reset --hard origin/main
else:
    print('🌐 Clonage propre du repository...')
    !git clone https://github.com/Albin0903/agario.git
    %cd agario

# Configuration du PYTHONPATH et installation des dépendances
os.environ['PYTHONPATH'] = f"{os.getcwd()}:{os.environ.get('PYTHONPATH', '')}"
!pip uninstall -y -q gym 2>/dev/null || true
!pip install -q -r requirements.txt
!apt-get install -qq -y ffmpeg
print('✅ Environnement et dépendances synchronisés avec succès.')


## 2. 🔍 Inspecteur de Modèles Google Drive (Inventaire Détaillé)
Cette cellule inspecte votre Google Drive, liste chaque fichier de modèle présent, sa taille, son palier de pas, et sélectionne automatiquement le meilleur modèle (champion 15M ou final).

In [ ]:
import os, glob, re, time
import pandas as pd

def extract_step(path):
    fname = os.path.basename(path)
    if 'final' in fname:
        return 999999999
    m = re.search(r'step_(\d+)', fname)
    return int(m.group(1)) if m else 0

search_dirs = [
    '/content/drive/MyDrive/agario_rl_backup_v2',
    '/content/drive/MyDrive/agario_rl_backup',
    'checkpoints/ppo',
    'models'
]

records = []
for sdir in search_dirs:
    if not os.path.exists(sdir): continue
    for f in glob.glob(os.path.join(sdir, '*.zip')) + glob.glob(os.path.join(sdir, '*.onnx')):
        st = os.stat(f)
        step_num = extract_step(f)
        records.append({
            'Nom': os.path.basename(f),
            'Type': 'PyTorch (.zip)' if f.endswith('.zip') else 'ONNX (.onnx)',
            'Palier Steps': f"{step_num:,}" if step_num < 999999999 else 'FINAL',
            'Taille (Mo)': round(st.st_size / (1024 * 1024), 2),
            'Date Modification': time.strftime('%d/%m/%Y %H:%M:%S', time.localtime(st.st_mtime)),
            'Dossier': sdir,
            'Chemin': f,
            '_step_raw': step_num
        })

if not records:
    raise FileNotFoundError("❌ Aucun fichier .zip ou .onnx trouvé sur Google Drive. Vérifiez que votre Drive est bien monté.")

df = pd.DataFrame(records)
df = df.sort_values(by='_step_raw', ascending=False).reset_index(drop=True)
display(df[['Nom', 'Type', 'Palier Steps', 'Taille (Mo)', 'Date Modification', 'Dossier']])

# Sélection prioritaire du modèle champion (15M ou final)
zip_models = df[df['Type'] == 'PyTorch (.zip)']
if not zip_models.empty:
    CHAMPION_MODEL = zip_models.iloc[0]['Chemin']
else:
    CHAMPION_MODEL = df.iloc[0]['Chemin']

print('\n' + '=' * 80)
print(f'🏆 MODÈLE CHAMPION SÉLECTIONNÉ : {CHAMPION_MODEL}')
print(f'💡 Vous pouvez modifier la variable CHAMPION_MODEL dans la cellule suivante si vous souhaitez tester un modèle spécifique.')
print('=' * 80)


## 3. 📊 Évaluation Scientifique du Modèle (Même Environnement Exact que l'Entraînement)
Cette cellule exécute une série de parties de test complètes dans `AgarEnv` avec les paramètres exacts de l'entraînement :
- Arène 1200x1200, 1500 pellets, 6 virus.
- **5 bots adversaires** (exactement comme à l'entraînement, et non 10).
- **Lissage de direction ($\alpha = 0.70$)** et **masquage de split sous 36 de masse** actifs.
- Calcul des moyennes de pic de masse, récompense, pellets et kills par épisode.

In [ ]:
# Vous pouvez spécifier manuellement un autre modèle ici si désiré :
# CHAMPION_MODEL = '/content/drive/MyDrive/agario_rl_backup_v2/ppo_step_15000000.zip'

# Exécution de l'évaluation sur 5 épisodes complets dans l'environnement d'entraînement
!python src/inference/eval_agent.py \
    --model "{CHAMPION_MODEL}" \
    --episodes 5


## 4. 🎬 Enregistrement du Match Replay HD (80 secondes) & Visionnage Direct
Génère une vidéo haute fidélité (2400 steps @ 30 FPS = 80 secondes) du modèle en action avec le lissage directionnel et le masquage de split.

In [ ]:
import os
from IPython.display import HTML, display
from base64 import b64encode

os.makedirs('recordings', exist_ok=True)
video_output = 'recordings/eval_champion_replay.mp4'

print(f'🎬 Enregistrement du replay HD avec le modèle : {CHAMPION_MODEL}')
!python src/inference/record_match.py \
    --model "{CHAMPION_MODEL}" \
    --output "{video_output}" \
    --steps 2400

# Sauvegarde miroir sur Google Drive
if os.path.exists(video_output) and os.path.exists('/content/drive/MyDrive/agario_rl_backup_v2'):
    !cp "{video_output}" /content/drive/MyDrive/agario_rl_backup_v2/eval_champion_replay.mp4
    print('📁 Replay sauvegardé sur Google Drive dans : agario_rl_backup_v2/eval_champion_replay.mp4')

# Affichage direct de la vidéo dans le Notebook
if os.path.exists(video_output):
    mp4_bytes = open(video_output, 'rb').read()
    data_url = 'data:video/mp4;base64,' + b64encode(mp4_bytes).decode()
    display(HTML(f'''
    <div style="text-align: center; margin: 15px 0;">
        <h3 style="color: #2c3e50;">🎮 Match Replay HD - Modèle Champion 15M</h3>
        <video width="850" height="480" controls autoplay loop style="border-radius: 8px; box-shadow: 0 4px 12px rgba(0,0,0,0.25);">
            <source src="{data_url}" type="video/mp4">
        </video>
        <p style="color: #666; font-size: 13px; margin-top: 8px;">
            Fichier : <code>eval_champion_replay.mp4</code> ({os.path.getsize(video_output) / 1_000_000:.1f} Mo)
        </p>
    </div>
    '''))
else:
    print('⚠️ Erreur : La vidéo n\'a pas pu être générée.')


## 5. ⚡ Export ONNX du Modèle Champion & Test d'Inférence
Exporte le modèle sélectionné vers le format haute performance ONNX et valide la parité.

In [ ]:
import os

os.makedirs('models', exist_ok=True)
onnx_output = 'models/model_champion.onnx'

# Export
!python src/inference/export_onnx.py \
    --model "{CHAMPION_MODEL}" \
    --output "{onnx_output}"

# Copie miroir sur Google Drive
if os.path.exists(onnx_output) and os.path.exists('/content/drive/MyDrive/agario_rl_backup_v2'):
    !cp "{onnx_output}" /content/drive/MyDrive/agario_rl_backup_v2/model_champion.onnx
    print('📁 Modèle ONNX sauvegardé sur Drive : agario_rl_backup_v2/model_champion.onnx')

# Test rapide d'inférence avec eval_agent sur le modèle ONNX
!python src/inference/eval_agent.py \
    --model "{onnx_output}" \
    --episodes 3
